# Análisis exploratorio del corpus TinyStories

Antes de empezar con el desarrollo del LLM es importante conocer el corpus que
se utilizará para su entrenamiento.

En este notebook se lleva a cabo un estudio del corpus de datos
[TinyStories](https://huggingface.co/datasets/roneneldan/TinyStories).

Este corpus contiene cuentos infantiles generados sintéticamente por GPT-3.5 y
GPT-4 con un vocabulario limitado.

Dadas las restricciones computacionales con las que se cuentan, este corpus es
ideal para entrenar el modelo de lenguaje.

## Carga del corpus

El corpus se encuentra disponible en [Hugging Face](https://huggingface.co/datasets/roneneldan/TinyStories).
Se utiliza la librería `datasets` para descargarlo fácilmente.

In [1]:
from datasets import concatenate_datasets, load_dataset

In [2]:
ds_dict = load_dataset("roneneldan/TinyStories")

Al cargar el corpus con `load_dataset()` los datos se dividen automáticamente en
los conjuntos de entrenamiento y validación.

Sin embargo, como el objetivo de este notebook no es entrenar el LLM, si no
explorar los datos, se agrupan en un único `DataSet` para facilitar su análisis.

In [3]:
ds_dict.shape

{'train': (2119719, 1), 'validation': (21990, 1)}

In [4]:
ds = concatenate_datasets(list(ds_dict.values()))

In [5]:
ds.shape

(2141709, 1)

## Análisis del corpus

Una vez el corpus ha sido cargado y se encuentra en un formato con el que
resulta cómodo trabajar, se procede a analizar el contenido del mismo:

In [6]:
print(f"El corpus tiene un total de {ds.shape[0]} ejemplos")

El corpus tiene un total de 2141709 ejemplos


El corpus contiene un gran número de ejemplos haciendo muy costoso, en términos
de cómputo, recorrerlo.

Para reducir el tiempo de computo, el corpus se recorrerá una única vez y, con
ayuda de las clases que se definen a continuación, se recopilará información
relevante sobre el corpus.

Se utiliza el tokenizer Tiktoken con el encoding `gpt2` por ser el mismo que se
utilizará en el LLM.

In [7]:
from collections import Counter
import math
import nltk
import numpy as np
import tiktoken

nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/jexposit/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [8]:
class TokenStats:
    def __init__(self) -> None:
        self.tokenizer = tiktoken.get_encoding("gpt2")
        self.min_tokens = float("inf")
        self.max_tokens = 0
        self.lengths = []
        self.total_tokens = 0
        self.total_chars = 0
        self.num_examples = 0

    def analyze_example(self, example: str) -> None:
        tokens = self.tokenizer.encode(example)
        num_tokens = len(tokens)

        self.min_tokens = min(self.min_tokens, num_tokens)
        self.max_tokens = max(self.max_tokens, num_tokens)
        self.lengths.append(num_tokens)
        self.total_tokens += num_tokens
        self.total_chars += len(example)
        self.num_examples += 1

    @property
    def mean_tokens(self) -> float:
        return self.total_tokens / self.num_examples

    @property
    def tokens_per_char(self):
        return self.total_tokens / self.total_chars

    def percentile(self, n):
        return np.percentile(self.lengths, n)

In [9]:
class VocabularyStats:
    def __init__(self) -> None:
        self.tokenizer = tiktoken.get_encoding("gpt2")
        self.total_tokens = 0
        self.token_counter = Counter()

        self.bigram_counter = Counter()
        self.trigram_counter = Counter()
        self.total_bigrams = 0
        self.total_trigrams = 0

    def analyze_example(self, example: str) -> None:
        tokens = self.tokenizer.encode(example)
        num_tokens = len(tokens)

        self.total_tokens += num_tokens
        self.token_counter.update(tokens)

        bigrams = list(zip(tokens, tokens[1:]))
        trigrams = list(zip(tokens, tokens[1:], tokens[2:]))

        self.bigram_counter.update(bigrams)
        self.trigram_counter.update(trigrams)

        self.total_bigrams += len(bigrams)
        self.total_trigrams += len(trigrams)

    @property
    def unique_tokens(self) -> int:
        return len(self.token_counter)
    
    @property
    def new_token_ratio(self):
        return self.unique_tokens / self.total_tokens

    @property
    def entropy(self):
        total = sum(self.token_counter.values())
        return -sum((c/total) * math.log2(c/total) for c in self.token_counter.values())

    @property
    def bigram_repetition_rate(self):
        repeated = sum(c for c in self.bigram_counter.values() if c > 1)
        return repeated / self.total_bigrams if self.total_bigrams else 0

    @property
    def trigram_repetition_rate(self):
        repeated = sum(c for c in self.trigram_counter.values() if c > 1)
        return repeated / self.total_trigrams if self.total_trigrams else 0

    def most_common_tokens(self, n: int = 10):
        most_common = []

        stopwords = nltk.corpus.stopwords.words('english')
        punctuation = ["'", "\"", ".", ",", ";", ":", "?", "!", ""]

        for token, count in self.token_counter.most_common():
            decoded_token = self.tokenizer.decode([token]).strip().lower()

            if decoded_token in punctuation:
                continue

            if decoded_token in stopwords:
                continue

            most_common.append((decoded_token, count))
            if len(most_common) == n:
                break

        return most_common

In [10]:
token_stats = TokenStats()
vocabulary_stats = VocabularyStats()

for example in ds["text"]:
    token_stats.analyze_example(example)
    vocabulary_stats.analyze_example(example)

In [11]:
print("Métricas de tokens:")
print(f" - Total:  El corpus tiene un total de {token_stats.total_tokens} tokens")
print(f" - Mínimo: El ejemplo con menor número de tokens tiene {token_stats.min_tokens} tokens")
print(f" - Máximo: El ejemplo con mayor número de tokens tiene {token_stats.max_tokens} tokens")
print(f" - Media:  La media de tokens por ejemplo es {token_stats.mean_tokens:.2f}")
print(f" - P5:     El percentil  5 es {token_stats.percentile(5):.2f} tokens")
print(f" - P50:    El percentil 50 es {token_stats.percentile(50):.2f} tokens")
print(f" - P95:    El percentil 95 es {token_stats.percentile(95):.2f} tokens")
print(f" - Tokens por letra: Una letra representa {token_stats.tokens_per_char:.2f} tokens")

print("\nMétricas de vocabulario:")
print(f" - Total: Hay {vocabulary_stats.unique_tokens} tokens únicos en el corpus")
print(f" - Entropía: {vocabulary_stats.entropy:.2f}")
print(f" - Ratio de tokens nuevos: El {vocabulary_stats.new_token_ratio * 100:.2f}% de tokens son nuevos")
print(f" - Repetición bigramas: {vocabulary_stats.bigram_repetition_rate * 100:.2f}%")
print(f" - Repetición trigramas: {vocabulary_stats.trigram_repetition_rate * 100:.2f}%")
print(f" - Tokens más comunes:")
for token, number in vocabulary_stats.most_common_tokens():
    print(f"   · {token} - {number}")


Métricas de tokens:
 - Total:  El corpus tiene un total de 476616445 tokens
 - Mínimo: El ejemplo con menor número de tokens tiene 0 tokens
 - Máximo: El ejemplo con mayor número de tokens tiene 1269 tokens
 - Media:  La media de tokens por ejemplo es 222.54
 - P5:     El percentil  5 es 130.00 tokens
 - P50:    El percentil 50 es 191.00 tokens
 - P95:    El percentil 95 es 455.00 tokens
 - Tokens por letra: Una letra representa 0.25 tokens

Métricas de vocabulario:
 - Total: Hay 29284 tokens únicos en el corpus
 - Entropía: 8.60
 - Ratio de tokens nuevos: El 0.01% de tokens son nuevos
 - Repetición bigramas: 99.82%
 - Repetición trigramas: 98.17%
 - Tokens más comunes:
   · said - 3631469
   · day - 2698769
   · lily - 2383469
   · mom - 2220406
   · 's - 2208034
   · time - 1827702
   · little - 1823969
   · happy - 1744667
   · saw - 1660336
   · big - 1657134


## Conclusiones

Durante el análisis del corpus el primer dato que se valora es su tamaño.
Es un corpus grande compuesto por más de dos millones de historias infantiles
escritas en inglés.

A continuación se hace un análisis del corpus en función de sus tokens.
El corpus cuenta con más de 476 millones de tokens. La longitud media de los
ejemplos es de ~222 tokens teniendo la mayoría de los ejemplos entre 130 tokens
(percentil 5) y 455 tokens (percentil 95). Esto indica que la mayoría de los
ejemplos son cortos y relativamente homogéneos en longitud.

Esto sugiere que el entrenamiento será eficiente pues hay poco desperdicio por
padding. Además, la mayoría de los ejemplos caben en contextos estándar.

En cuanto a la "tokenización", el ratio de 0.25 tokens por carácter es el
esperable para el idioma inglés y refleja una codificación eficiente del texto:  
https://help.openai.com/en/articles/4936856-what-are-tokens-and-how-to-count-them

Finalmente, se lleva a cabo un análisis del corpus desde el punto de vista del
vocabulario.

El corpus presenta un número moderado de tokens únicos moderado (~29000) y una
entropía de 8.60, lo que indica una diversidad de vocabulario limitada.

Al dato de la entropía se le suma un ratio de tokens nuevos extremadamente bajo,
de tan solo el 0.01%, y una alta repetición de bigramas y trigramas.

Todo esto indica que tanto el vocabulario como la estructura de los ejemplos es
sencilla y que el corpus está fuertemente dominado por la repetición.

El listado de tokens más frecuentes respalda la afirmación anterior, mostrando
palabras sencillas.

Por todo esto, el corpus es ideal para entrenamiento eficiente y rápido de
modelos pequeños.